In [5]:
import urllib.request
import duckdb

DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
PARQUET_FILE = "yellow_tripdata_2024-01.parquet"
DB_FILE = "taxi.db"

In [6]:
def setup_database():
    con = duckdb.connect(DB_FILE)

    if not os.path.exists(PARQUET_FILE):
        urllib.request.urlretrieve(DATA_URL, PARQUET_FILE)

    con.execute(f"""
        CREATE TABLE IF NOT EXISTS trips AS
        SELECT * FROM '{PARQUET_FILE}'
    """)

    return con.execute("SELECT COUNT(*) FROM trips").fetchone()[0]

In [12]:
import os 
row_count = setup_database()
row_count

2964624

In [1]:
class SQLTools:

    def __init__(self, db_file):
        self.con = duckdb.connect(db_file)

    def get_schema(self) -> str:
        rows = self.con.execute("DESCRIBE trips").fetchall()
        return "\n".join([f"{r[0]} ({r[1]})" for r in rows])

    def run_sql(self, query: str) -> str:
        result = self.con.execute(query)
        columns = [d[0] for d in result.description]
        rows = result.fetchmany(50)

        output = " | ".join(columns) + "\n"
        output += "-" * 60 + "\n"
        for row in rows:
            output += " | ".join(str(x) for x in row) + "\n"

        return output

In [2]:
class ToolKit:

    def __init__(self, tool_instance):
        self.tool_instance = tool_instance

    def get_tools(self):
        return [
            self.tool_instance.get_schema,
            self.tool_instance.run_sql,
        ]

In [3]:
from pydantic import BaseModel

class SQLResult(BaseModel):
    sql_query: str
    result_text: str
    row_count: int

In [ ]:
from pydantic_ai import Agent

sql_tools = SQLTools(DB_FILE)
toolkit = ToolKit(sql_tools)

SYSTEM_PROMPT = """
You are a SQL Agent for NYC Taxi dataset.

IMPORTANT: The table name is "trips" - NOT "taxi_rides" or any other name.

Rules:
1. Always call get_schema FIRST to see available columns.
2. The table is named "trips" - always use this name in queries.
3. Generate correct SQL using only available columns from the schema.
4. Return structured JSON:
   - sql_query
   - result_text
   - row_count
"""

agent = Agent(
    model="gpt-4o-mini",
    output_type=SQLResult,
    tools=toolkit.get_tools(),
    system_prompt=SYSTEM_PROMPT,
)

In [8]:
def collect_tools(messages):
    tool_calls = []
    for msg in messages:
        if msg.tool_calls:
            for call in msg.tool_calls:
                tool_calls.append(call.function.name)
    return tool_calls

In [ ]:

SYSTEM_PROMPT_V2 = """You are a SQL Agent for the NYC Yellow Taxi dataset stored in duckdb.

DATABASE SCHEMA:
- Table name: "trips" (this is the ONLY table)
- The table contains NYC taxi trip data from 2024

CRITICAL RULES:
1. ALWAYS start by calling get_schema() to see the exact column names
2. NEVER use table names like 'taxi_rides', 'taxi_data', 'yellow_trips', etc.
3. ALWAYS use the table name "trips" in all SQL queries
4. Only use column names that exist in the schema
5. Generate correct SQL using DuckDB syntax
6. When asked about average, use AVG() function
7. When asked about passengers, look for columns like 'passenger_count' or similar

Return a SQLResult with:
- sql_query: the exact SQL query you ran
- result_text: the query results
- row_count: number of rows returned"""

agent = Agent(
    model="gpt-4o-mini",
    output_type=SQLResult,
    tools=toolkit.get_tools(),
    system_prompt=SYSTEM_PROMPT_V2,
)


print("Running query with new agent...")
result_test = await agent.run("What's the average trip distance for rides with 2 passengers?")
print(f"SQL Query: {result_test.output.sql_query}")
print(f"Result: {result_test.output.result_text}")


Running query with new agent...
SQL Query: SELECT AVG(trip_distance) AS average_trip_distance FROM trips WHERE passenger_count = 2
Result: 3.7827640377879197


In [15]:
result_q2 = await agent.run("What's the average trip distance for rides with 2 passengers?")

result_q2.output

SQLResult(sql_query='SELECT AVG(trip_distance) AS average_trip_distance \nFROM trips \nWHERE passenger_count = 2;', result_text='3.7827640377879184', row_count=1)

In [ ]:
print("Result from the SQL Agent:")
print(f"SQL Query: {result_q2.output.sql_query}")
print(f"\nQuery Result:\n{result_q2.output.result_text}")
print(f"Row Count: {result_q2.output.row_count}")


import re
match = re.search(r'(\d+\.\d+)', result_q2.output.result_text)
if match:
    avg_distance = float(match.group(1))
    print(f"\n✓ Average trip distance for 2-passenger rides: {avg_distance}")


In [ ]:

result_q3 = await agent.run("How many trips had more than 5 passengers?")
print(f"Q3 Query: {result_q3.output.sql_query}")
print(f"Q3 Result:\n{result_q3.output.result_text}")
print(f"Q3 Row Count: {result_q3.output.row_count}")


Q3 Query: SELECT COUNT(*) AS trip_count FROM trips WHERE passenger_count > 5;
Q3 Result:
22413
Q3 Row Count: 1


In [ ]:

result_q4 = await agent.run("What is the most common payment type?")
print("Q4 Query:", result_q4.output.sql_query)
print("Q4 Result:", result_q4.output.result_text)
print("Q4 Row Count:", result_q4.output.row_count)


Q4 Query: SELECT payment_type, COUNT(*) as payment_count 
FROM trips 
GROUP BY payment_type 
ORDER BY payment_count DESC 
LIMIT 1;
Q4 Result: 1 | 2319046
Q4 Row Count: 1


In [ ]:

result_q5 = await agent.run("Which hour of the day has the highest average fare amount?")
print("Q5 Query:", result_q5.output.sql_query)
print("Q5 Result:", result_q5.output.result_text)


Q5 Query: SELECT HOUR(tpep_pickup_datetime) AS hour_of_day, AVG(fare_amount) AS average_fare
FROM trips
GROUP BY hour_of_day
ORDER BY average_fare DESC
LIMIT 1;
Q5 Result: 5 | 26.619918460882563


In [ ]:

result_q6_zero = await agent.run("How many trips had zero passengers recorded?")
print("Q6 (zero passengers) Query:", result_q6_zero.output.sql_query)
print("Q6 (zero passengers) Result:", result_q6_zero.output.result_text)
print("Column used:", "passenger_count" if "passenger_count" in result_q6_zero.output.sql_query else "OTHER")


Q6 (zero passengers) Query: SELECT COUNT(*) AS zero_passenger_trips FROM trips WHERE passenger_count = 0;
Q6 (zero passengers) Result: 31465
Column used: passenger_count


In [ ]:

test_calls = [
    "test_q3_trips_more_than_5_passengers",  
    "test_q4_tool_call_order",  
    "test_q5_llm_judge_highest_fare_hour",  
    "test_q6_avg_tip_credit_card",  
    "test_q6_zero_passengers",  
]


num_notebook_calls = 5  
num_test_calls = 7  
total_calls = num_notebook_calls + num_test_calls


cost_per_call = 0.015  
estimated_total_cost = total_calls * cost_per_call

print(f"Estimated API Calls: {total_calls}")
print(f"Estimated cost per call: ${cost_per_call:.4f}")
print(f"Estimated Total Cost: ${estimated_total_cost:.4f}")
print()
print("Cost ranges:")
if estimated_total_cost < 0.05:
    print("Less than $0.05")
elif estimated_total_cost < 0.20:
    print("$0.05 - $0.20")
elif estimated_total_cost < 1.00:
    print("$0.20 - $1.00")
else:
    print("More than $1.00")


Estimated API Calls: 12
Estimated cost per call: $0.0150
Estimated Total Cost: $0.1800

Cost ranges:
✓ $0.05 - $0.20
